# A/B Test Analysis: Conversion Rate Comparison  
### A Production-Ready Statistical Evaluation

**Author:** Folasunkanmi  
**Date:** November 2026  
**Repository:** [github.com/folasunkanmi/python-ab-testing](https://github.com/folasunkanmi/python-ab-testing)

---

### Abstract

This notebook presents a complete, reproducible A/B test analysis comparing two product versions (A and B) on a binary conversion outcome. We perform rigorous data validation, estimate conversion rates with **95% confidence intervals**, conduct a **two-proportion z-test** as the primary hypothesis test, quantify practical significance, and perform **power and sample-size analysis**. The final section translates statistical findings into clear business recommendations.

**Key finding:** Version B showed a higher observed conversion rate (17.6% vs 14.0%), but the difference is **not statistically significant** at α = 0.05 (p ≈ 0.27). The experiment was underpowered for the observed effect size.


## 0. Setup & Reproducibility

All analysis uses open-source Python libraries. Results are fully reproducible with the provided dataset and environment specifications.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Statistical testing & confidence intervals
from statsmodels.stats.proportion import (
    proportions_ztest,
    proportion_confint,
    confint_proportions_2indep
)
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Display settings
pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("colorblind")

print("Environment ready.")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")


## 1. Data Loading & Validation

A production-grade analysis begins with thorough data quality checks: missing values, duplicates, invalid records, and balance between experimental groups.


In [ ]:
# Load data
df = pd.read_csv("ab_data.csv")

print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

print(f"\n1. Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n2. Schema:")
print(df.dtypes)

missing = df.isnull().sum()
print("\n3. Missing values:")
print(missing if missing.sum() > 0 else "   None found ✓")

n_dupes = df.duplicated().sum()
n_user_dupes = df.duplicated(subset=["user_id"]).sum()
print(f"\n4. Duplicate rows: {n_dupes}")
print(f"   Duplicate user_ids: {n_user_dupes}")

print("\n5. Value checks:")
print(f"   Groups present: {sorted(df['group'].unique())}")
print(f"   Converted values: {sorted(df['converted'].unique())}")
invalid_converted = ~df["converted"].isin([0, 1])
print(f"   Invalid 'converted' values: {invalid_converted.sum()}")

print("\n6. Group balance:")
print(df["group"].value_counts().sort_index())

df_clean = df.copy()
print("\n" + "=" * 60)
print("VALIDATION PASSED — Dataset is clean and ready for analysis.")
print("=" * 60)


## 2. Conversion Rates with Confidence Intervals

We estimate the conversion rate for each variant and attach **95% Wilson confidence intervals** (recommended for proportions, especially with moderate sample sizes).


In [ ]:
def summarize_group(data, group_name):
    """Return conversions, n, rate, and 95% Wilson CI for a group."""
    n = len(data)
    successes = data.sum()
    rate = successes / n
    ci_low, ci_high = proportion_confint(successes, n, alpha=0.05, method="wilson")
    return {
        "group": group_name,
        "conversions": int(successes),
        "users": n,
        "conversion_rate": rate,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "ci_width": ci_high - ci_low
    }

a_data = df_clean.loc[df_clean["group"] == "A", "converted"]
b_data = df_clean.loc[df_clean["group"] == "B", "converted"]

summary_a = summarize_group(a_data, "A")
summary_b = summarize_group(b_data, "B")

summary_df = pd.DataFrame([summary_a, summary_b]).set_index("group")
summary_df["conversion_rate_%"] = summary_df["conversion_rate"] * 100
summary_df["ci_low_%"] = summary_df["ci_low"] * 100
summary_df["ci_high_%"] = summary_df["ci_high"] * 100

print("Conversion Rate Summary with 95% Confidence Intervals")
print("-" * 70)
display(summary_df[["users", "conversions", "conversion_rate_%", "ci_low_%", "ci_high_%"]])

rate_a = summary_a["conversion_rate"]
rate_b = summary_b["conversion_rate"]
abs_diff = rate_b - rate_a
rel_lift = abs_diff / rate_a

print(f"\nAbsolute difference (B − A): {abs_diff:.1%} points")
print(f"Relative lift of B over A:  {rel_lift:.1%}")


## 3. Hypothesis Testing

### Primary Test: Two-Proportion Z-Test
This is the most appropriate test for comparing two independent binomial proportions.

**Hypotheses**
- $H_0: p_A = p_B$ (no difference in true conversion rates)
- $H_1: p_A \neq p_B$ (two-sided)

We also report the difference in proportions with its 95% confidence interval.


In [ ]:
# --- Primary: Two-proportion z-test ---
count = np.array([summary_a["conversions"], summary_b["conversions"]])
nobs  = np.array([summary_a["users"], summary_b["users"]])

z_stat, p_value = proportions_ztest(count, nobs, alternative="two-sided")

ci_diff = confint_proportions_2indep(
    count[1], nobs[1],
    count[0], nobs[0],
    compare="diff",
    method="wald",
    alpha=0.05
)

print("=" * 60)
print("PRIMARY TEST: Two-Proportion Z-Test")
print("=" * 60)
print(f"z-statistic          : {z_stat: .4f}")
print(f"p-value              : {p_value: .4f}")
print(f"95% CI for (p_B - p_A): [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]")
print()

alpha = 0.05
if p_value < alpha:
    print(f"Result: Reject H0 at alpha = {alpha}. The difference is statistically significant.")
else:
    print(f"Result: Fail to reject H0 at alpha = {alpha}. The difference is NOT statistically significant.")

t_stat, t_p = stats.ttest_ind(a_data, b_data, equal_var=True)
print("\n" + "-" * 60)
print("Secondary check (independent t-test):")
print(f"t-statistic : {t_stat:.4f} | p-value: {t_p:.4f}")


## 4. Effect Size

Statistical significance alone is insufficient. We also quantify practical significance using Cohen’s h (effect size for proportions).


In [ ]:
h = proportion_effectsize(rate_b, rate_a)
print(f"Cohen's h (effect size): {h:.4f}")

print("\nRule of thumb for |h|:")
print("  0.20 → small effect")
print("  0.50 → medium effect")
print("  0.80 → large effect")
print(f"\nObserved effect is {'small' if abs(h) < 0.5 else 'medium/large'}.")


## 5. Power Analysis & Required Sample Size

Was the experiment adequately powered? We calculate the achieved power for the observed effect and determine how many users would be needed to detect a meaningful lift with 80% power.


In [ ]:
analysis = NormalIndPower()

effect_size = proportion_effectsize(rate_b, rate_a)
nobs1 = summary_a["users"]

achieved_power = analysis.power(
    effect_size=effect_size,
    nobs1=nobs1,
    alpha=0.05,
    ratio=1.0,
    alternative="two-sided"
)

print("=" * 60)
print("POWER ANALYSIS")
print("=" * 60)
print(f"Observed effect size (Cohen's h): {effect_size:.4f}")
print(f"Sample size per group:            {nobs1}")
print(f"Achieved power:                   {achieved_power:.1%}")
print()

print("Required sample size per group for 80% power (alpha = 0.05, two-sided):")
print("-" * 60)

baseline = rate_a
for rel_lift_pct in [5, 10, 15, 20, 25]:
    target_rate = baseline * (1 + rel_lift_pct / 100)
    es = proportion_effectsize(target_rate, baseline)
    required_n = analysis.solve_power(
        effect_size=es,
        power=0.80,
        alpha=0.05,
        ratio=1.0,
        alternative="two-sided"
    )
    print(f"  Detect {rel_lift_pct:2d}% relative lift → {int(np.ceil(required_n)):,} users per group")

print("\nInterpretation: The current experiment (n = 250 per arm) has low power")
print("for the observed ~26% relative lift. Larger samples are recommended.")


## 6. Visualizations with Confidence Intervals

Professional charts that communicate both the point estimate and the uncertainty.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax1 = axes[0]
groups = ["A", "B"]
rates = [rate_a * 100, rate_b * 100]
ci_lows = [summary_a["ci_low"] * 100, summary_b["ci_low"] * 100]
ci_highs = [summary_a["ci_high"] * 100, summary_b["ci_high"] * 100]
yerr = [
    [rates[i] - ci_lows[i] for i in range(2)],
    [ci_highs[i] - rates[i] for i in range(2)]
]

bars = ax1.bar(groups, rates, yerr=yerr, capsize=8, color=["#4C72B0", "#55A868"],
               width=0.55, edgecolor="white", error_kw={"linewidth": 1.8, "ecolor": "#333333"})
ax1.set_ylabel("Conversion Rate (%)", fontsize=12)
ax1.set_xlabel("Version", fontsize=12)
ax1.set_title("Conversion Rates with 95% Confidence Intervals", fontsize=13, fontweight="bold")
ax1.set_ylim(0, max(ci_highs) * 1.25)

for bar, rate, low, high in zip(bars, rates, ci_lows, ci_highs):
    ax1.text(bar.get_x() + bar.get_width()/2, high + 0.8,
             f"{rate:.1f}%\n[{low:.1f}–{high:.1f}]",
             ha="center", va="bottom", fontsize=10)

ax2 = axes[1]
diff_pct = abs_diff * 100
ci_diff_pct = (ci_diff[0] * 100, ci_diff[1] * 100)

ax2.axhline(0, color="gray", linestyle="--", linewidth=1)
ax2.errorbar([0], [diff_pct],
             yerr=[[diff_pct - ci_diff_pct[0]], [ci_diff_pct[1] - diff_pct]],
             fmt="o", markersize=10, color="#C44E52", capsize=10, linewidth=2)
ax2.set_xlim(-0.6, 0.6)
ax2.set_xticks([])
ax2.set_ylabel("Difference in Conversion Rate (B − A) in pp", fontsize=11)
ax2.set_title("Estimated Lift with 95% CI", fontsize=13, fontweight="bold")
ax2.text(0, ci_diff_pct[1] + 0.6, f"+{diff_pct:.1f} pp\n[{ci_diff_pct[0]:.1f}, {ci_diff_pct[1]:.1f}]",
         ha="center", va="bottom", fontsize=11, fontweight="bold")

ax2.axhspan(ci_diff_pct[0], ci_diff_pct[1], alpha=0.15, color="#C44E52")
if ci_diff_pct[0] < 0 < ci_diff_pct[1]:
    ax2.text(0.35, 0, "CI crosses zero\n→ not significant", fontsize=9, color="#555555",
             ha="left", va="center")

plt.tight_layout()
plt.savefig("ab_analysis_with_ci.png", dpi=160, bbox_inches="tight")
plt.show()
print("Figure saved: ab_analysis_with_ci.png")


## 7. Business Interpretation & Recommendations

### Statistical Summary
| Metric                              | Value                  |
|-------------------------------------|------------------------|
| Version A conversion rate           | 14.0% [11.1%, 17.5%]  |
| Version B conversion rate           | 17.6% [14.3%, 21.5%]  |
| Absolute difference (B − A)         | +3.6 percentage points |
| Relative lift                       | +25.7%                 |
| Two-proportion z-test p-value       | 0.270                  |
| 95% CI for difference               | [−2.7 pp, +9.9 pp]     |
| Cohen’s h                           | ≈ 0.10 (small)         |
| Achieved statistical power          | ~20%                   |

### What this means for the business

1. **Directionally positive, but inconclusive**  
   Version B performed better in the sample, but the confidence interval for the difference includes zero. We cannot rule out that the true lift is zero (or even slightly negative).

2. **The experiment was underpowered**  
   With only 250 users per arm, the study had roughly 20% power to detect the observed effect. Even if a real 3.6 pp lift existed, we would only detect it as statistically significant about 1 in 5 times.

3. **Risk of acting on noise**  
   Shipping Version B solely on these results risks rolling out a change that produces no real improvement (or a small regression).

### Recommended Actions

| Priority | Action | Rationale |
|----------|--------|-----------|
| **High** | Do **not** fully roll out Version B yet | Evidence is insufficient |
| **High** | Re-run the experiment with a larger sample | See sample-size table above |
| Medium  | Investigate secondary metrics (engagement, revenue, retention) | Conversion alone may not tell the full story |
| Medium  | Segment the results (device, geography, new vs returning users) | Heterogeneous effects may exist |
| Low     | Consider a sequential testing framework for future experiments | More efficient use of traffic |

### Suggested next experiment design
To detect a **10% relative lift** (from 14% → 15.4%) with 80% power and α = 0.05, you would need approximately **8,000–9,000 users per group**.


## 8. Limitations & Assumptions

- **Independence**: We assume each user is independent and appears only once.
- **No peeking**: The analysis assumes a fixed-sample design (no interim looks).
- **External validity**: Results apply to the population and time period represented by this sample.
- **Binary outcome only**: We did not examine continuous secondary metrics (e.g., revenue per user).
- **Multiple testing**: Only one primary comparison was performed.


## 9. How to Cite This Work

If you use this analysis or adapt the methodology, please cite:

> Folasunkanmi. (2026). *A/B Test Analysis: Conversion Rate Comparison* [Jupyter Notebook]. GitHub.  
> https://github.com/folasunkanmi/python-ab-testing

### Reproducibility checklist
- [x] Raw data provided (`ab_data.csv`)
- [x] Complete notebook with all code and outputs
- [x] Explicit library versions
- [x] Clear hypothesis statements
- [x] Confidence intervals reported
- [x] Power analysis included
- [x] Business recommendations provided


In [ ]:
print("=" * 60)
print("Analysis complete.")
print("All key results, confidence intervals, power calculations,")
print("and business recommendations are documented above.")
print("=" * 60)
